# Spring-temperature sensitivity and ERA5-Land climate correlations

Per-range regressions of the annual onset anomaly on ERA5-Land anomalies (all variables and
months), the spring-temperature sensitivity (histograms, per-range regressions,
per-range scatterplots) and its relation to the elevation lapse rate. The
numbers themselves (slopes, r, p, n) are written by `pipeline/scripts/range_metrics.py` into
`results/<version>/mountain_range_metrics.csv`; this notebook plots them.

The per-range scatter sweep (the map insets) and the composite temperature-sensitivity world map are built by
`spring_temperature_sensitivity_composite_figure.ipynb` (one notebook per composite figure since 2026-09).

In [ ]:
import os
import textwrap

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from shapely.geometry import box

from gsro_analysis import aggregate, paths, plotting, settings, stats
from gsro_analysis.plotting import (
    count_valid_obs, create_gmba_exists_gdf, create_mini_polar_plot,
    get_sorted_ranges, label_angle, major_tick_radii,
    plot_anomaly_heatmap_panels, plot_mountain_range_anomalies,
    plot_mountain_ranges_on_map, rgrid_labels, rgrid_labels_blank,
    rgrid_vals, wrap_labels)
from gsro_analysis.stats import build_anomalies_df

In [ ]:
config = settings.load_config()  # the dataset version lives in settings.CONFIG_FILE
gmba_gdf = aggregate.load_gmba()

mountains_full = aggregate.open_aggregate('mountain_ranges', config.version)
mountains_ds = stats.prepare_mountain_ranges(mountains_full)   # CHILI collapsed, the analyses' thresholds, mean anomaly added

# ERA5-Land anomaly zonal means per range (pipeline/scripts/era5_zonal.py, merged by reduce_partials):
# (range, water_year, month) for the 8 variables; add seasonal means and energy-balance sums
from gsro_analysis import era5
climate_vars = [v for v in era5.VARIABLES if v in mountains_ds]
assert climate_vars, "no ERA5 variables in the cube - run pipeline/scripts/era5_zonal.py, then reduce_partials.py"
climate_ds = stats.seasonal_means(mountains_ds[climate_vars])
climate_ds['total_radiation_sum'] = climate_ds['surface_solar_radiation_downwards_sum'] + climate_ds['surface_thermal_radiation_downwards_sum']
climate_ds['total_turbulent_flux_sum'] = climate_ds['surface_sensible_heat_flux_sum'] + climate_ds['surface_latent_heat_flux_sum']
climate_ds['total_surface_energy_sum'] = climate_ds['total_radiation_sum'] + climate_ds['total_turbulent_flux_sum']
mountains_ds = xr.merge([mountains_ds.drop_vars(climate_vars), climate_ds])
climate_vars = list(climate_ds.data_vars)

# per-range metrics (lapse rates, sensitivity, anomalies) are computed ONCE by
# pipeline/scripts/range_metrics.py into the one results table; this notebook plots them
metrics = stats.range_metrics_table(config.version).set_index('name')
gmba_stats_gdf = stats.range_metrics_gdf(config.version, gmba_gdf)   # GMBA polygons + the metrics, joined on read
mountains_ds

## Data coverage per range and year

In [ ]:
# fraction of a range's median-pixel count with data in each water year (the 0.1 rule masks range-years below it)
mountains_proportion_of_pixels_each_WY_to_10yr_median_da = (
    mountains_ds['runoff_onset_n'].sum(['elevation', 'aspect'])
    / mountains_ds['runoff_onset_median_n'].sum(['elevation', 'aspect']))
f, ax = plt.subplots(figsize=(10, 30))
mountains_proportion_of_pixels_each_WY_to_10yr_median_da.plot.imshow(ax=ax)

## Regressions of the onset anomaly on every ERA5 variable and month

In [ ]:
mountains_linreg_ds = stats.climate_regressions(mountains_ds, climate_vars)   # (range, month, param) per variable
mountains_linreg_ds

### Heatmaps (corr / slope / p-value — three deliberate variants)

In [ ]:
# var = 'surface_thermal_radiation_downwards_sum'
# var = 'surface_solar_radiation_downwards_sum'
# var = 'surface_sensible_heat_flux_sum'
# var = 'surface_latent_heat_flux_sum'
# var = 'snow_depth_water_equivalent'
# var ='snowfall_sum'
# var = 'dewpoint_temperature_2m'
var ='temperature_2m'
#var = 'total_surface_energy_sum'
#var = 'total_radiation_sum'
#var = 'total_turbulent_flux_sum'

anomalies_df = pd.DataFrame(
    mountains_linreg_ds[var].sel(param='corr').values,
    index=mountains_linreg_ds[var].mountain_range.values,
    columns=mountains_linreg_ds[var].month.values
)

anomalies_df['continent'] = mountains_linreg_ds['continent'].values
anomalies_df['latitude'] = mountains_linreg_ds['centroid_latitude'].values
anomalies_df['name'] = mountains_linreg_ds['mountain_range'].values

# Create figure
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 20))

# Plot Americas
americas_df = anomalies_df[anomalies_df['continent'].isin(['North America', 'South America'])]
na_df = americas_df[americas_df['continent']=='North America'].sort_values('latitude', ascending=False)
sa_df = americas_df[americas_df['continent']=='South America'].sort_values('latitude', ascending=False)
americas_df = pd.concat([na_df, sa_df])
# Filter rows with enough observations
americas_df = americas_df[americas_df[mountains_linreg_ds[var].month.values].apply(count_valid_obs, axis=1) > 5]

sns.heatmap(americas_df.set_index('name')[mountains_linreg_ds[var].month.values],
            ax=ax1, cmap='RdBu', vmin=-1, vmax=1, cbar_kws={'label': 'Days'},linewidths=1,square=True,cbar=False)
#wrap_labels(ax1)

#ax1.axhline(len(na_df), color='black', linewidth=2)
ax1.set_title('North America and South America')
#ax1.grid(True)

# Plot Europe and Africa
euraf_df = anomalies_df[anomalies_df['continent'].isin(['Europe', 'Africa'])]
eu_df = euraf_df[euraf_df['continent']=='Europe'].sort_values('latitude', ascending=False)
af_df = euraf_df[euraf_df['continent']=='Africa'].sort_values('latitude', ascending=False)
euraf_df = pd.concat([eu_df, af_df])
euraf_df = euraf_df[euraf_df[mountains_linreg_ds[var].month.values].apply(count_valid_obs, axis=1) > 5]

sns.heatmap(euraf_df.set_index('name')[mountains_linreg_ds[var].month.values],
            ax=ax2, cmap='RdBu', vmin=-1, vmax=1, cbar_kws={'label': 'Days'},linewidths=1,square=True,cbar=False)
#ax2.axhline(len(eu_df), color='black', linewidth=2)
#wrap_labels(ax2)

ax2.set_title('Europe and Africa')
#ax2.grid(True)

# Plot Asia and Oceania
asoc_df = anomalies_df[anomalies_df['continent'].isin(['Asia', 'Oceania', 'Australia'])]
asia_df = asoc_df[asoc_df['continent']=='Asia'].sort_values('latitude', ascending=False)
oc_df = asoc_df[asoc_df['continent'].isin(['Oceania', 'Australia'])].sort_values('latitude', ascending=False)
asoc_df = pd.concat([asia_df, oc_df])
asoc_df = asoc_df[asoc_df[mountains_linreg_ds[var].month.values].apply(count_valid_obs, axis=1) > 5]

sns.heatmap(asoc_df.set_index('name')[mountains_linreg_ds[var].month.values],
            ax=ax3, cmap='RdBu', vmin=-1, vmax=1, cbar_kws={'label': 'Days'},linewidths=1,square=True)
#ax3.axhline(len(asia_df), color='black', linewidth=2)
#wrap_labels(ax3)

ax3.set_title('Asia and Oceania')
#ax3.grid(True)

# Common settings
for ax in [ax1, ax2, ax3]:
    ax.set_facecolor('darkgrey')
    ax.set_xlabel('month')
    ax.set_ylabel('')

fig.suptitle(f'snowmelt runoff onset timing anomaly correlation with ERA5-Land {var} anomaly')

# fig.savefig(paths.figdir('mountain_ranges', config.version) / 'snowmelt_onset_anomalies_by_mountain_range.png')

#plt.tight_layout()
#plt.show()

In [ ]:
# var = 'surface_thermal_radiation_downwards_sum'
# var = 'surface_solar_radiation_downwards_sum'
# var = 'surface_sensible_heat_flux_sum'
# var = 'surface_latent_heat_flux_sum'
# var = 'snow_depth_water_equivalent'
# var ='snowfall_sum'
#var = 'dewpoint_temperature_2m'
var ='temperature_2m'
#var = 'total_surface_energy_sum'
#var = 'total_radiation_sum'

anomalies_df = pd.DataFrame(
    mountains_linreg_ds[var].sel(param='slope').values,
    index=mountains_linreg_ds[var].mountain_range.values,
    columns=mountains_linreg_ds[var].month.values
)

anomalies_df['continent'] = mountains_linreg_ds['continent'].values
anomalies_df['latitude'] = mountains_linreg_ds['centroid_latitude'].values
anomalies_df['name'] = mountains_linreg_ds['mountain_range'].values

# Create figure
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 20))

# Plot Americas
americas_df = anomalies_df[anomalies_df['continent'].isin(['North America', 'South America'])]
na_df = americas_df[americas_df['continent']=='North America'].sort_values('latitude', ascending=False)
sa_df = americas_df[americas_df['continent']=='South America'].sort_values('latitude', ascending=False)
americas_df = pd.concat([na_df, sa_df])
# Filter rows with enough observations
americas_df = americas_df[americas_df[mountains_linreg_ds[var].month.values].apply(count_valid_obs, axis=1) > 5]

sns.heatmap(americas_df.set_index('name')[mountains_linreg_ds[var].month.values],
            ax=ax1, cmap='RdBu', vmin=-10, vmax=10, cbar_kws={'label': 'Days'},linewidths=1,square=True,cbar=False)
#wrap_labels(ax1)

#ax1.axhline(len(na_df), color='black', linewidth=2)
ax1.set_title('North America and South America')
#ax1.grid(True)

# Plot Europe and Africa
euraf_df = anomalies_df[anomalies_df['continent'].isin(['Europe', 'Africa'])]
eu_df = euraf_df[euraf_df['continent']=='Europe'].sort_values('latitude', ascending=False)
af_df = euraf_df[euraf_df['continent']=='Africa'].sort_values('latitude', ascending=False)
euraf_df = pd.concat([eu_df, af_df])
euraf_df = euraf_df[euraf_df[mountains_linreg_ds[var].month.values].apply(count_valid_obs, axis=1) > 5]

sns.heatmap(euraf_df.set_index('name')[mountains_linreg_ds[var].month.values],
            ax=ax2, cmap='RdBu', vmin=-10, vmax=10, cbar_kws={'label': 'Days'},linewidths=1,square=True,cbar=False)
#ax2.axhline(len(eu_df), color='black', linewidth=2)
#wrap_labels(ax2)

ax2.set_title('Europe and Africa')
#ax2.grid(True)

# Plot Asia and Oceania
asoc_df = anomalies_df[anomalies_df['continent'].isin(['Asia', 'Oceania', 'Australia'])]
asia_df = asoc_df[asoc_df['continent']=='Asia'].sort_values('latitude', ascending=False)
oc_df = asoc_df[asoc_df['continent'].isin(['Oceania', 'Australia'])].sort_values('latitude', ascending=False)
asoc_df = pd.concat([asia_df, oc_df])
asoc_df = asoc_df[asoc_df[mountains_linreg_ds[var].month.values].apply(count_valid_obs, axis=1) > 5]

sns.heatmap(asoc_df.set_index('name')[mountains_linreg_ds[var].month.values],
            ax=ax3, cmap='RdBu', vmin=-10, vmax=10, cbar_kws={'label': 'Days'},linewidths=1,square=True)
#ax3.axhline(len(asia_df), color='black', linewidth=2)
#wrap_labels(ax3)

ax3.set_title('Asia and Oceania')
#ax3.grid(True)

# Common settings
for ax in [ax1, ax2, ax3]:
    ax.set_facecolor('darkgrey')
    ax.set_xlabel('month')
    ax.set_ylabel('')

fig.suptitle(f'snowmelt runoff onset timing anomaly correlation with ERA5-Land {var} anomaly')

# fig.savefig(paths.figdir('mountain_ranges', config.version) / 'snowmelt_onset_anomalies_by_mountain_range.png')

#plt.tight_layout()
#plt.show()

In [ ]:
# var = 'surface_thermal_radiation_downwards_sum'
# var = 'surface_solar_radiation_downwards_sum'
# var = 'surface_sensible_heat_flux_sum'
# var = 'surface_latent_heat_flux_sum'
# var = 'snow_depth_water_equivalent'
# var ='snowfall_sum'
#var = 'dewpoint_temperature_2m'
var ='temperature_2m'
#var = 'total_surface_energy_sum'
#var = 'total_radiation_sum'

anomalies_df = pd.DataFrame(
    mountains_linreg_ds[var].sel(param='pval').values,
    index=mountains_linreg_ds[var].mountain_range.values,
    columns=mountains_linreg_ds[var].month.values
)

anomalies_df['continent'] = mountains_linreg_ds['continent'].values
anomalies_df['latitude'] = mountains_linreg_ds['centroid_latitude'].values
anomalies_df['name'] = mountains_linreg_ds['mountain_range'].values

# Create figure
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 20))

# Plot Americas
americas_df = anomalies_df[anomalies_df['continent'].isin(['North America', 'South America'])]
na_df = americas_df[americas_df['continent']=='North America'].sort_values('latitude', ascending=False)
sa_df = americas_df[americas_df['continent']=='South America'].sort_values('latitude', ascending=False)
americas_df = pd.concat([na_df, sa_df])
# Filter rows with enough observations
americas_df = americas_df[americas_df[mountains_linreg_ds[var].month.values].apply(count_valid_obs, axis=1) > 5]

sns.heatmap(americas_df.set_index('name')[mountains_linreg_ds[var].month.values],
            ax=ax1, cmap='seismic', vmin=0, vmax=0.1, cbar_kws={'label': 'Days'},linewidths=1,square=True,cbar=False)
#wrap_labels(ax1)

#ax1.axhline(len(na_df), color='black', linewidth=2)
ax1.set_title('North America and South America')
#ax1.grid(True)

# Plot Europe and Africa
euraf_df = anomalies_df[anomalies_df['continent'].isin(['Europe', 'Africa'])]
eu_df = euraf_df[euraf_df['continent']=='Europe'].sort_values('latitude', ascending=False)
af_df = euraf_df[euraf_df['continent']=='Africa'].sort_values('latitude', ascending=False)
euraf_df = pd.concat([eu_df, af_df])
euraf_df = euraf_df[euraf_df[mountains_linreg_ds[var].month.values].apply(count_valid_obs, axis=1) > 5]

sns.heatmap(euraf_df.set_index('name')[mountains_linreg_ds[var].month.values],
            ax=ax2, cmap='seismic', vmin=0, vmax=0.1, cbar_kws={'label': 'Days'},linewidths=1,square=True,cbar=False)
#ax2.axhline(len(eu_df), color='black', linewidth=2)
#wrap_labels(ax2)

ax2.set_title('Europe and Africa')
#ax2.grid(True)

# Plot Asia and Oceania
asoc_df = anomalies_df[anomalies_df['continent'].isin(['Asia', 'Oceania', 'Australia'])]
asia_df = asoc_df[asoc_df['continent']=='Asia'].sort_values('latitude', ascending=False)
oc_df = asoc_df[asoc_df['continent'].isin(['Oceania', 'Australia'])].sort_values('latitude', ascending=False)
asoc_df = pd.concat([asia_df, oc_df])
asoc_df = asoc_df[asoc_df[mountains_linreg_ds[var].month.values].apply(count_valid_obs, axis=1) > 5]

sns.heatmap(asoc_df.set_index('name')[mountains_linreg_ds[var].month.values],
            ax=ax3, cmap='seismic', vmin=0, vmax=0.1, cbar_kws={'label': 'Days'},linewidths=1,square=True)
#ax3.axhline(len(asia_df), color='black', linewidth=2)
#wrap_labels(ax3)

ax3.set_title('Asia and Oceania')
#ax3.grid(True)

# Common settings
for ax in [ax1, ax2, ax3]:
    ax.set_facecolor('darkgrey')
    ax.set_xlabel('month')
    ax.set_ylabel('')

fig.suptitle(f'snowmelt runoff onset timing anomaly pvalue with ERA5-Land {var} anomaly')

# fig.savefig(paths.figdir('mountain_ranges', config.version) / 'snowmelt_onset_anomalies_by_mountain_range.png')

#plt.tight_layout()
#plt.show()

### One range, all variables

In [ ]:
mountain_range = 'Bureya Region'
#mountain_range = 'Cordillera de la Costa'
# mountain_range = 'Cordillera Oriental (Northern Andes)' # looks dominated by turbulent fluxes!!!!
#mountain_range = 'Verkhoyansk Range'
mountain_range = 'Sierra Nevada'
mountain_range = 'Chersky Range' if 'Chersky Range' in mountains_ds.mountain_range.values else str(mountains_ds.mountain_range.values[0])
#mountain_range='Jura Mountains'
# mountain_range = 'Shanxi Mountains'
# mountain_range = 'Ordos Plateau'

# #mountain_range = 'Alashan Plateau'
# mountain_range = 'Yin Mountains'
# mountain_range = 'Yan Mountains'
# mountain_range = 'Arakan Mountains'
# mountain_range = 'Sierras Pampeanas'
mountain_unstacked_da = mountains_linreg_ds.sel(mountain_range=mountain_range,param='corr').to_dataarray()
mountain_unstacked_da['variable'] = mountain_unstacked_da['variable'].astype(str)


f,ax=plt.subplots(figsize=(6,6))
sns.heatmap(mountain_unstacked_da.to_pandas(), ax=ax, cmap='RdBu', vmin=-1, vmax=1, cbar_kws={'label': 'r'})
ax.set_title(f'{mountain_range}: correlation of the onset anomaly with each ERA5 anomaly')

In [ ]:
mountain_range = 'Bureya Region'
#mountain_range = 'Cordillera de la Costa'
# mountain_range = 'Cordillera Oriental (Northern Andes)' # looks dominated by turbulent fluxes!!!!
#mountain_range = 'Verkhoyansk Range'
mountain_range = 'Sierra Nevada'
mountain_range = 'Chersky Range' if 'Chersky Range' in mountains_ds.mountain_range.values else str(mountains_ds.mountain_range.values[0])
#mountain_range='Jura Mountains'
# mountain_range = 'Shanxi Mountains'
# mountain_range = 'Ordos Plateau'

# #mountain_range = 'Alashan Plateau'
# mountain_range = 'Yin Mountains'
# mountain_range = 'Yan Mountains'
# mountain_range = 'Arakan Mountains'
# mountain_range = 'Sierras Pampeanas'
mountain_unstacked_da = mountains_linreg_ds.sel(mountain_range=mountain_range,param='pval').to_dataarray()
mountain_unstacked_da['variable'] = mountain_unstacked_da['variable'].astype(str)


f,ax=plt.subplots(figsize=(6,6))
sns.heatmap(mountain_unstacked_da.to_pandas(), ax=ax, cmap='seismic', vmin=0, vmax=0.1, cbar_kws={'label': 'p-value'})
ax.set_title(f'{mountain_range}: p-value of the regression on each ERA5 anomaly')

In [ ]:
import sklearn.linear_model
mountain_range = 'Alaska Range' if 'Alaska Range' in mountains_ds.mountain_range.values else str(mountains_ds.mountain_range.values[0])
#mountain_range = 'Sierra Nevada'
#mountain_range = 'Cordillera Oriental (Northern Andes)' 
var = 'surface_sensible_heat_flux_sum'
var='temperature_2m'


f,ax=plt.subplots(dpi=300)
era5_anoms = mountains_ds[var].sel(mountain_range=mountain_range).sel(month='spring_months_mean')#.sel(month=['spring_month_1', 'spring_month_2','spring_month_3']).mean(dim='month')
#temp_anoms = mountains_ds['spring_temp_mean_anomaly'].sel(mountain_range=mountain_range)
melt_anoms = mountains_ds['runoff_onset_mean_anomaly'].sel(mountain_range=mountain_range)
water_years = era5_anoms['water_year']

combined_df = pd.DataFrame({
    'era5_anom': era5_anoms.values,
    'melt_anom': melt_anoms.values,
    'water_year': water_years.values
}).dropna()

# plot temperature anomalies on x axis, melt anomalies on y axis,color should be water year
plot = ax.scatter(combined_df['era5_anom'], combined_df['melt_anom'], c=combined_df['water_year'], cmap='viridis')

f.colorbar(plot, label='Water year')

# vertical and horizontal dashed lines at 0
ax.axhline(0, color='gray', linestyle='--')
ax.axvline(0, color='gray', linestyle='--')


#ax.set_xlim([-3.2,3.2])
ax.set_ylim([-35,35])

# create a grid
ax.grid(True, which='both', linestyle='--', linewidth=0.5)
# ax.set_xticks([-3,-2,-1,0,1,2,3])
# ax.set_yticks([-30,-20,-10,0,10,20,30])

ax.set_xlabel(f'10-year (WY2015-WY2024) Spring {var} Anomaly [°C]')
ax.set_ylabel('10-year (WY2015-WY2024) Snowmelt Runoff Onset Anomaly [days]')

# calculate thiel-sen regression line, plot line, add slope and correlation to title
reg = sklearn.linear_model.TheilSenRegressor().fit(
    combined_df['era5_anom'].values.reshape(-1, 1),
    combined_df['melt_anom'].values
)

slope = reg.coef_[0]
intercept = reg.intercept_

r_squared = reg.score(
    combined_df['era5_anom'].values.reshape(-1, 1),
    combined_df['melt_anom'].values
)
corr = np.sqrt(r_squared)

ax.set_title(f'{mountain_range}\nCorrelation: {corr:.2f} | Slope: {slope:.1f} days/°C')
x_smooth = np.linspace(era5_anoms.min().values, era5_anoms.max().values, 100)
expected_melt_anom = slope * x_smooth + intercept
ax.plot(x_smooth, expected_melt_anom, color='red', linestyle='--')

## Spring-temperature sensitivity per range

In [ ]:
# facet: spring 2 m temperature anomaly vs onset anomaly for every range (OLS)
import scipy.stats as sps
var, months = 'temperature_2m', 'spring_months_mean'
ranges = mountains_ds['mountain_range'].values
cols = 6
rows = (len(ranges) // cols) + (len(ranges) % cols > 0)
f, axs = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), layout='constrained', squeeze=False, dpi=150)
for ax, mountain_range in zip(axs.flat, ranges):
    era5_anoms = mountains_ds[var].sel(mountain_range=mountain_range, month=months)
    melt_anoms = mountains_ds['runoff_onset_mean_anomaly'].sel(mountain_range=mountain_range)
    combined_df = pd.DataFrame({'era5_anom': era5_anoms.values, 'melt_anom': melt_anoms.values,
                                'water_year': era5_anoms['water_year'].values}).dropna()
    ax.scatter(combined_df['era5_anom'], combined_df['melt_anom'], c=combined_df['water_year'], cmap='YlGnBu',
               vmin=config.water_years[0], vmax=config.water_years[-1], edgecolors='black', s=50)
    ax.axhline(0, color='black', linestyle=':'); ax.axvline(0, color='black', linestyle=':')
    ax.set_xlabel(f'spring ERA5-Land {var} anomaly [°C]'); ax.set_ylabel('runoff onset anomaly [days]')
    ax.set_xlim([-3.2, 3.2]); ax.set_ylim([-35, 35]); ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    title = '\n'.join(textwrap.wrap(mountain_range, 40))
    if len(combined_df) >= 3:
        res = sps.linregress(combined_df['era5_anom'], combined_df['melt_anom'])
        x_smooth = np.linspace(combined_df['era5_anom'].min(), combined_df['era5_anom'].max(), 20)
        ax.plot(x_smooth, res.slope * x_smooth + res.intercept, color='red', linestyle='--')
        ax.set_title(f'{title}\nn: {len(combined_df)} | r: {res.rvalue:.2f} | slope: {res.slope:.1f} days/°C')
    else:
        ax.set_title(f'{title}\nNo Fit')
for ax in axs.flat[len(ranges):]:
    ax.set_visible(False)

In [ ]:
# take the previous plot and create a facet plot of all mountain ranges in range_metadata
var = 'temperature_2m'

num_ranges = len(mountains_ds['mountain_range'])
cols = 6
rows = (num_ranges // cols) + (num_ranges % cols > 0)
f,axs=plt.subplots(rows,cols,figsize=(5*cols,4*rows),layout='constrained',squeeze=True,dpi=200)#sharex=True,sharey=True,dpi=300)

for i,mountain_range in enumerate(mountains_ds['mountain_range'].values):
    #print(mountain_range)
    ax = axs.flatten()[i]
    era5_anoms = mountains_ds[var].sel(mountain_range=mountain_range).sel(month='spring_months_mean')#.sel(month=['spring_month_1', 'spring_month_2','spring_month_3']).mean(dim='month')
    melt_anoms = mountains_ds['runoff_onset_mean_anomaly'].sel(mountain_range=mountain_range)
    water_years = era5_anoms['water_year']
    
    combined_df = pd.DataFrame({
        'era5_anom': era5_anoms.values,
        'melt_anom': melt_anoms.values,
        'water_year': water_years.values
    }).dropna()

    # plot temperature anomalies on x axis, melt anomalies on y axis,color should be water year
    plot = ax.scatter(combined_df['era5_anom'],combined_df['melt_anom'],c=combined_df['water_year'],cmap='YlGnBu',vmin=config.water_years[0],vmax=config.water_years[-1], edgecolors='black',s=50)

    # vertical and horizontal dashed lines at 0
    ax.axhline(0, color='black', linestyle=':')
    ax.axvline(0, color='black', linestyle=':')

    # ax.set_xlabel('10-year (WY2015-WY2024) Spring Temperature Anomaly [°C]')
    # ax.set_ylabel('10-year (WY2015-WY2024) Snowmelt Runoff Onset Anomaly [days]')

    ax.set_xlabel(f'10-yr spring ERA5-Land {var} anomaly [°C]')
    ax.set_ylabel('10-yr runoff onset anomaly [days]')
    
    if var == 'temperature_2m':
        ax.set_xlim([-3.2,3.2])
    ax.set_ylim([-35,35])

    # create a grid
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    #ax.set_xticks([-3,-2,-1,0,1,2,3])
    #ax.set_yticks([-30,-20,-10,0,10,20,30])

    # calculate and print correlation coefficient and line of best fit

    try:
        #slope, intercept = np.polyfit(combined_df['era5_anom'], combined_df['melt_anom'], 1)
        
        slope, intercept, low_slope, high_slope = stats.mstats.theilslopes(
            combined_df['melt_anom'], combined_df['era5_anom'], 0.95
        )
        
        
        
  
        
        
        n = len(combined_df)

        title = mountain_range
        if len(title)>40:
            title = "\n".join(textwrap.wrap(title, 40))

        ax.set_title(f'{title}\nn: {len(combined_df)} | slope: {slope:.1f} days/°C | low: {low_slope:.1f} | high: {high_slope:.1f}')
        print(f'mountain range: {mountain_range}, n: {len(combined_df)}, slope:{slope:.2f}, low slope: {low_slope:.2f} days/°C, high slope: {high_slope:.2f} days/°C')

        x_smooth = np.linspace(combined_df['era5_anom'].min(), combined_df['era5_anom'].max(), 20)
        expected_melt_anom = slope * x_smooth + intercept
        ax.plot(x_smooth, expected_melt_anom, color='red', linestyle='--')
        # now plot the low and high slope as dashed lines
        expected_melt_anom_low = low_slope * x_smooth + intercept
        expected_melt_anom_high = high_slope * x_smooth + intercept
        ax.plot(x_smooth, expected_melt_anom_low, color='blue', linestyle=':')
        ax.plot(x_smooth, expected_melt_anom_high, color='blue', linestyle=':')

        # if mountain_range in gmba_gdf['MapName'].values:
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'anomaly_slope'] = slope
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'anomaly_corr'] = corr
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'anomaly_n'] = n
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'snowmelt_lapse_rate_per_100m'] = snowmelt_elevation_stats_ds['lapse_rate_per_100m'].sel(mountain_range=mountain_range).values
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'snowmelt_lapse_rate_corr'] = snowmelt_elevation_stats_ds['correlation'].sel(mountain_range=mountain_range).values
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'snowmelt_lapse_rate_n'] = snowmelt_elevation_stats_ds['n'].sel(mountain_range=mountain_range).values
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'total_pixels_in_range'] = total_pixels_in_each_range_da.sel(mountain_range=mountain_range).values
        # else:
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'anomaly_slope'] = slope
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'anomaly_corr'] = corr
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'anomaly_n'] = n
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'snowmelt_lapse_rate_per_100m'] = snowmelt_elevation_stats_ds['lapse_rate_per_100m'].sel(mountain_range=mountain_range).values
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'snowmelt_lapse_rate_corr'] = snowmelt_elevation_stats_ds['correlation'].sel(mountain_range=mountain_range).values
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'snowmelt_lapse_rate_n'] = snowmelt_elevation_stats_ds['n'].sel(mountain_range=mountain_range).values
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'total_pixels_in_range'] = total_pixels_in_each_range_da.sel(mountain_range=mountain_range).values

    except Exception as e:
        print(f"Could not calculate fit for {mountain_range}: {e}")
        ax.set_title(f'{mountain_range}\nNo Fit')
        
#f.tight_layout()

f.savefig(paths.figdir('mountain_ranges', config.version) / 'mountain_range_spring_temp_vs_runoff_onset_anomaly_scatterplots_theil_sen.png', dpi=200)

In [ ]:
import sklearn.linear_model
import pymannkendall as mk
import scipy
# take the previous plot and create a facet plot of all mountain ranges in range_metadata
var = 'temperature_2m'

num_ranges = len(mountains_ds['mountain_range'])
cols = 6
rows = (num_ranges // cols) + (num_ranges % cols > 0)
f,axs=plt.subplots(rows,cols,figsize=(5*cols,4*rows),layout='constrained',squeeze=True,dpi=200)#sharex=True,sharey=True,dpi=300)

for i,mountain_range in enumerate(mountains_ds['mountain_range'].values):
    #print(mountain_range)
    ax = axs.flatten()[i]
    era5_anoms = mountains_ds[var].sel(mountain_range=mountain_range).sel(month='spring_months_mean')#.sel(month=['spring_month_1', 'spring_month_2','spring_month_3']).mean(dim='month')
    melt_anoms = mountains_ds['runoff_onset_mean_anomaly'].sel(mountain_range=mountain_range)
    water_years = era5_anoms['water_year']
    
    combined_df = pd.DataFrame({
        'era5_anom': era5_anoms.values,
        'melt_anom': melt_anoms.values,
        'water_year': water_years.values
    }).dropna()

    # plot temperature anomalies on x axis, melt anomalies on y axis,color should be water year
    plot = ax.scatter(combined_df['era5_anom'],combined_df['melt_anom'],c=combined_df['water_year'],cmap='YlGnBu',vmin=config.water_years[0],vmax=config.water_years[-1], edgecolors='black',s=50)

    # vertical and horizontal dashed lines at 0
    ax.axhline(0, color='black', linestyle=':')
    ax.axvline(0, color='black', linestyle=':')

    # ax.set_xlabel('10-year (WY2015-WY2024) Spring Temperature Anomaly [°C]')
    # ax.set_ylabel('10-year (WY2015-WY2024) Snowmelt Runoff Onset Anomaly [days]')

    ax.set_xlabel(f'10-yr spring ERA5-Land {var} anomaly [°C]')
    ax.set_ylabel('10-yr runoff onset anomaly [days]')
    
    if var == 'temperature_2m':
        ax.set_xlim([-3.2,3.2])
    ax.set_ylim([-35,35])

    # create a grid
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    #ax.set_xticks([-3,-2,-1,0,1,2,3])
    #ax.set_yticks([-30,-20,-10,0,10,20,30])

    # calculate and print correlation coefficient and line of best fit

    try:
        #slope, intercept = np.polyfit(combined_df['era5_anom'], combined_df['melt_anom'], 1)
        reg = sklearn.linear_model.TheilSenRegressor().fit(
            combined_df['era5_anom'].values.reshape(-1, 1),
            combined_df['melt_anom'].values
        )
        slope = reg.coef_[0]
        intercept = reg.intercept_
        r_squared = reg.score(
            combined_df['era5_anom'].values.reshape(-1, 1),
            combined_df['melt_anom'].values
        )

        corr = np.sqrt(r_squared)
        
        kendalltau = scipy.stats.kendalltau(combined_df['era5_anom'], combined_df['melt_anom'])
        #corr = np.corrcoef(combined_df['era5_anom'], combined_df['melt_anom'])[0, 1]
        # order points for the mann-kendall test
        mannkendall_df = combined_df.sort_values('era5_anom')
        
        trend, h, p, z, Tau, s, var_s, mk_slope, mk_intercept = mk.original_test(mannkendall_df['melt_anom'])
        
        # add trend and p-value as annotations on the plot
        ax.annotate(f'Trend: {trend}\n p-value: {p:.3f}\n MK Slope: {mk_slope:.2f} days/°C\n kendall Tau: {kendalltau.statistic:.2f},\np-value: {kendalltau.pvalue:.3f}',
                    xy=(0.05, 0.95), xycoords='axes fraction',
                    fontsize=8, ha='left', va='top',
                    bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.5))
        
        n = len(combined_df)

        title = mountain_range
        if len(title)>40:
            title = "\n".join(textwrap.wrap(title, 40))

        ax.set_title(f'{title}\nn: {len(combined_df)} | r: {corr:.2f} | slope: {slope:.1f} days/°C')
        print(f'mountain range: {mountain_range}, n: {len(combined_df)}, r:{corr:.2f}, slope: {slope:.2f} days/°C')
        print(f'mannkendall test: {trend}, p-value: {p:.3f}, slope: {mk_slope:.2f} days/°C')

        x_smooth = np.linspace(combined_df['era5_anom'].min(), combined_df['era5_anom'].max(), 20)
        expected_melt_anom = slope * x_smooth + intercept
        ax.plot(x_smooth, expected_melt_anom, color='red', linestyle='--')

        # if mountain_range in gmba_gdf['MapName'].values:
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'anomaly_slope'] = slope
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'anomaly_corr'] = corr
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'anomaly_n'] = n
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'snowmelt_lapse_rate_per_100m'] = snowmelt_elevation_stats_ds['lapse_rate_per_100m'].sel(mountain_range=mountain_range).values
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'snowmelt_lapse_rate_corr'] = snowmelt_elevation_stats_ds['correlation'].sel(mountain_range=mountain_range).values
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'snowmelt_lapse_rate_n'] = snowmelt_elevation_stats_ds['n'].sel(mountain_range=mountain_range).values
        #     gmba_gdf.loc[gmba_gdf['MapName'] == mountain_range, 'total_pixels_in_range'] = total_pixels_in_each_range_da.sel(mountain_range=mountain_range).values
        # else:
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'anomaly_slope'] = slope
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'anomaly_corr'] = corr
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'anomaly_n'] = n
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'snowmelt_lapse_rate_per_100m'] = snowmelt_elevation_stats_ds['lapse_rate_per_100m'].sel(mountain_range=mountain_range).values
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'snowmelt_lapse_rate_corr'] = snowmelt_elevation_stats_ds['correlation'].sel(mountain_range=mountain_range).values
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'snowmelt_lapse_rate_n'] = snowmelt_elevation_stats_ds['n'].sel(mountain_range=mountain_range).values
        #     gmba_gdf.loc[gmba_gdf['Level_04'] == mountain_range, 'total_pixels_in_range'] = total_pixels_in_each_range_da.sel(mountain_range=mountain_range).values

    except Exception as e:
        print(f"Could not calculate fit for {mountain_range}: {e}")
        ax.set_title(f'{mountain_range}\nNo Fit')
        
#f.tight_layout()

## Sensitivity histograms, significance

In [ ]:
# show histograms for slope, correlation, and p-value
f,ax=plt.subplots(1,3,figsize=(18,3),dpi=300)
gmba_stats_gdf['anomaly_slope'].dropna().hist(bins=40, ax=ax[0])
ax[0].set_title('Spring temp sensitivity [days/°C]')
ax[0].set_xlabel('Sensitivity [days/°C]')
ax[0].set_ylabel('Frequency')
ax[0].axvline(0, color='black', linestyle='--')

gmba_stats_gdf['anomaly_corr'].dropna().hist(bins=40, ax=ax[1])
ax[1].set_title('Spring temp sensitivity correlation')
ax[1].set_xlabel('Pearson correlation (r)')
ax[1].set_ylabel('Frequency')
ax[1].axvline(0, color='black', linestyle='--')

gmba_stats_gdf['anomaly_pval'].dropna().hist(bins=40, ax=ax[2])
ax[2].set_title('Spring temp sensitivity p-value')
ax[2].set_xlabel('p-value')
ax[2].set_ylabel('Frequency')
ax[2].axvline(0.05, color='red', linestyle='--')

f.savefig(paths.figdir('mountain_ranges', config.version) / 'snowmelt_onset_spring_temp_sensitivity_histograms.png', bbox_inches='tight')

In [ ]:
# what percent of mountain ranges have p-value < 0.05
gmba_sensitivity_analysis_gdf = gmba_stats_gdf[~gmba_stats_gdf['anomaly_pval'].isna()]
num_ranges_in_sensitivity = len(gmba_sensitivity_analysis_gdf)
significant_ranges = len(gmba_sensitivity_analysis_gdf[gmba_sensitivity_analysis_gdf['anomaly_pval'] <= 0.05])
print(f"Total mountain ranges: {num_ranges_in_sensitivity}")
print(f"Significant ranges (p < 0.05): {significant_ranges} ({significant_ranges/num_ranges_in_sensitivity:.2%})")

In [ ]:
# plot ranges with p-value < 0.05 as blue, >=0.05 as gray on a world map
gmba_sensitivity_analysis_gdf['significant'] = gmba_sensitivity_analysis_gdf['anomaly_pval'] <= 0.05

# drop arctic cordillera because it is misleading, we dont have the entire area
gmba_sensitivity_analysis_gdf = gmba_sensitivity_analysis_gdf[gmba_sensitivity_analysis_gdf['MapName'] != 'Arctic Cordillera']

f,ax=plt.subplots(figsize=(15,10),dpi=300,subplot_kw={'projection': ccrs.Robinson()})
gmba_sensitivity_analysis_gdf.plot(column='significant', ax=ax, legend=True,
               legend_kwds={'labels': [f'p >= 0.05 [count={gmba_sensitivity_analysis_gdf[~gmba_sensitivity_analysis_gdf["significant"].values].shape[0]}]', f'p < 0.05 [count={gmba_sensitivity_analysis_gdf[gmba_sensitivity_analysis_gdf["significant"].values].shape[0]}]']},
               cmap='PiYG', edgecolor='black', linewidth=0.01, transform=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND, facecolor='darkgrey', alpha=1)
ax.add_feature(cfeature.OCEAN, facecolor='dimgray', alpha=1)
        
ax.set_title(f'Mountain ranges with significant runoff onset sensitivity to spring air temperature (p < 0.05)')

## Sensitivity vs elevation lapse rate (results table from range_metrics.py)

In [ ]:
# sensitivity vs lapse rate: the columns of the metrics table under the names the cells below use
era5_anomaly_df = (metrics.reset_index()[['name', 'anomaly_slope', 'anomaly_corr', 'anomaly_n', 'snowmelt_lapse_rate_per_100m',
                                          'snowmelt_lapse_rate_corr', 'snowmelt_lapse_rate_n', 'total_pixels_in_range']]
                   .rename(columns={'name': 'MapName', 'anomaly_slope': 'temp_anomaly_slope', 'anomaly_corr': 'temp_anomaly_corr',
                                    'anomaly_n': 'temp_anomaly_n'}).dropna(thresh=4))
era5_anomaly_df

In [ ]:
# find the 2.5 and 97.5 percentiles of temp_anomaly_slope
lower_percentile = era5_anomaly_df['temp_anomaly_slope'].quantile(0.05)
upper_percentile = era5_anomaly_df['temp_anomaly_slope'].quantile(0.95)
print(f"5th Percentile: {lower_percentile:.2f}")
print(f"95th Percentile: {upper_percentile:.2f}")

In [ ]:
import scipy.stats as stats
# now create a scatter plot of temp_anomaly_slope vs snowmelt_lapse_rate_per_100m
f, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    era5_anomaly_df['temp_anomaly_slope'],
    era5_anomaly_df['snowmelt_lapse_rate_per_100m'],
    c=era5_anomaly_df['temp_anomaly_corr'],
    cmap='RdBu',
    edgecolor='black',
    s=20,
    vmin=-1,
    vmax=1,
)
ax.legend(*scatter.legend_elements(), title='Temp Anomaly Correlation (r)')
# label axis
ax.set_xlabel('Temperature Anomaly Sensitivity (days/°C)')
ax.set_ylabel('Snowmelt Runoff Onset Elevation Lapse Rate (days/100m)')
ax.set_title('Mountain Range Temperature Sensitivity vs Snowmelt Runoff Onset Elevation Lapse Rate')



# do a quick linear regression and plot line of best fit with equation and r value
slope, intercept, r_value, p_value, std_err = stats.linregress(
    era5_anomaly_df['temp_anomaly_slope'],
    era5_anomaly_df['snowmelt_lapse_rate_per_100m'],
)
x_smooth = np.linspace(era5_anomaly_df['temp_anomaly_slope'].min(), era5_anomaly_df['temp_anomaly_slope'].max(), 100)
y_smooth = slope * x_smooth + intercept
ax.plot(x_smooth, y_smooth, color='red', linestyle='--', label='Line of Best Fit')
ax.text(0.05, 0.95,
        f'y = {slope:.2f}x + {intercept:.2f}\nr = {r_value:.2f}',
        transform=ax.transAxes,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.5)
)

# label the points
for i, row in era5_anomaly_df.iterrows():
    ax.text(
        row['temp_anomaly_slope'],
        row['snowmelt_lapse_rate_per_100m'],
        row['MapName'],
        fontsize=6,
        ha='right'
    )
    
ax.set_xlim([-14, 10])
ax.set_ylim([-5, 10])

## Per-range scatterplots and the world map — built elsewhere

The per-range scatterplot sweep (the map insets) and the composite temperature-sensitivity world map are built by [`spring_temperature_sensitivity_composite_figure.ipynb`](spring_temperature_sensitivity_composite_figure.ipynb) (one notebook per composite figure since 2026-09). The Equal-Earth maps below are exploratory.

## Sensitivity maps

In [ ]:
f,ax=plt.subplots(figsize=(12,5),subplot_kw={'projection': ccrs.EqualEarth()},dpi=300)

# Base map
ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.75)
ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.75)
ax.add_feature(cfeature.COASTLINE, linewidth=0.45)
    
gmba_stats_gdf.plot(column='anomaly_corr', cmap='Spectral', legend=True, vmin=-1,vmax=1,ax=ax,transform=ccrs.PlateCarree(),edgecolor='black',linewidth=0.1, label="Runoff anomaly correlation with spring temp anomaly", legend_kwds={'label': "Runoff anomaly correlation with spring temp anomaly",'pad':0.01})


ax.set_extent([-200, 200, -62, 82], crs=ccrs.PlateCarree())
ax.set_xlim(left=-1.25E7)

In [ ]:
f,ax=plt.subplots(figsize=(12,5),subplot_kw={'projection': ccrs.EqualEarth()},dpi=300)

# Base map
ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.75)
ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.75)
#ax.add_feature(cfeature.COASTLINE, linewidth=0.45)
    
gmba_stats_gdf.plot(column='anomaly_slope', cmap='YlOrRd_r', legend=True, vmin=-15,vmax=0,ax=ax,transform=ccrs.PlateCarree(),edgecolor='black',linewidth=0.1, label="Runoff anomaly correlation with spring temp anomaly", legend_kwds={'label': "Runoff anomaly correlation with spring temp anomaly",'pad':0.01})

ax.coastlines(resolution='10m', linewidth=0.2)

gl = ax.gridlines(draw_labels=True)
gl.top_labels = False
gl.right_labels = False

ax.set_extent([-200, 200, 20, 72], crs=ccrs.PlateCarree())
ax.set_xlim(left=-1.25E7)

In [ ]:
#f,ax=plt.subplots(figsize=(12,5),subplot_kw={'projection': ccrs.AlbersEqualArea(central_longitude=-96, central_latitude=45, standard_parallels=(30,60))},dpi=300)
f,ax=plt.subplots(figsize=(12,5),subplot_kw={'projection': ccrs.AlbersEqualArea(central_longitude=90, central_latitude=50, standard_parallels=(30,70))},dpi=300)


# Base map
ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.75)
ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.75)
#ax.add_feature(cfeature.COASTLINE, linewidth=0.45)
ax.coastlines(resolution='10m', linewidth=0.2)

gmba_stats_gdf.plot(column='anomaly_slope', cmap='YlOrRd_r', vmin=-15,vmax=0, legend=True, ax=ax,transform=ccrs.PlateCarree(),edgecolor='black',linewidth=0.1, label="Runoff onset timing shift\nfor each degree increase in spring temp anomaly", legend_kwds={'label': "Runoff onset timing shift\nfor each degree increase in spring temp anomaly",'pad':0.01})

gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.bottom_labels = False
gl.left_labels = False
gl.top_labels = False
gl.right_labels = False

ax.set_extent([60, 175, 25, 72], crs=ccrs.PlateCarree())
#ax.set_xlim(left=-1.25E7)

## Correlation vs slope

In [ ]:
# NOW PLOT CORRELATION VS SLOPE
f,ax=plt.subplots(figsize=(15,7))
slope = gmba_stats_gdf['anomaly_slope']
corr = gmba_stats_gdf['anomaly_corr']
names = gmba_stats_gdf['MapName']
latitude = np.abs(gmba_stats_gdf.geometry.centroid.y)
plot = ax.scatter(slope,corr,c=latitude,cmap='gist_rainbow',vmin=0,vmax=70, edgecolors='black')
f.colorbar(plot, label='Latitude')

ax.set_xlabel('Slope [days/°C]')
ax.set_ylabel('Correlation Coefficient')
ax.set_title('Snowmelt Runoff Onset Anomaly vs Spring Temperature Anomaly\nCorrelation vs Sensitivity Slope')

# shade area based on correlation
ax.axhspan(0.5, 1, color='green', alpha=0.1)
ax.axhspan(-1, -0.5, color='red', alpha=0.1)
ax.axhspan(-0.5, 0.5, color='gray', alpha=0.1)

ax.axvline(0, color='gray', linestyle='--')

In [ ]:
# now plot latitude vs correlation, colored by slope
f,ax=plt.subplots(figsize=(12,7))
slope = gmba_stats_gdf['anomaly_slope']
corr = gmba_stats_gdf['anomaly_corr']
names = gmba_stats_gdf['MapName']
latitude = gmba_stats_gdf.geometry.centroid.y
plot = ax.scatter(latitude,corr,c=slope,cmap='coolwarm',vmin=-20,vmax=20, edgecolors='black')
f.colorbar(plot, label='Slope [days/°C]')

ax.axhspan(0.5, 1, color='green', alpha=0.1)
ax.axhspan(-1, -0.5, color='red', alpha=0.1)
ax.axhspan(-0.5, 0.5, color='gray', alpha=0.1)

ax.set_xlabel('Latitude')
ax.set_ylabel('Correlation Coefficient')
ax.set_title('Snowmelt Runoff Onset Anomaly vs Spring Temperature Anomaly\nLatitude vs Correlation')